# 🎙️ TurboVoiceCloner (One-Click Setup)
**Features:** Turbo Speed | Silence Remover | Auto-Push to GitHub | Anti-Disconnect

In [ ]:
# @title 🚀 Step 1: Initialize Environment & Anti-Disconnect
# @markdown यह सेल Python 3.10 सेटअप करेगा और ब्राउज़र को स्लीपिंग मोड से बचाएगा।

import time
from IPython.display import display, Javascript

# Anti-Disconnect Script (Sleeping mode preventer)
display(Javascript('''
function ClickConnect(){
  console.log("Keeping Colab Alive...");
  document.querySelector("colab-run-button").click()
}
setInterval(ClickConnect, 60000)
'''))

print("Installing Python 3.10 and Requirements... This is a one-time setup.")
!sudo apt-get update -y -q
!sudo apt-get install python3.10 python3.10-dev python3.10-distutils -y -q
!wget https://bootstrap.pypa.io/get-pip.py -q
!python3.10 get-pip.py -q

# Automatic Installation from requirements.txt
!python3.10 -m pip install -q -r https://raw.githubusercontent.com/shriramnag/TurboVoiceCloner/main/requirements.txt
print("✅ Everything Installed at Turbo High Speed!")

In [ ]:
# @title ⚙️ Step 2: Launch App
with open("app_fixed.py", "w") as f:
    f.write('''
import gradio as gr
from TTS.api import TTS
from pydub import AudioSegment, silence
import os

device = "cuda" if os.path.exists("/dev/nvidia0") else "cpu"
tts = TTS("tts_models/multilingual/multi-dataset/your_tts").to(device)

def process(text, ref, clean):
    out = "temp.wav"
    final = "final_voice.wav"
    tts.tts_to_file(text=text, speaker_wav=ref, language="en", file_path=out)
    if clean:
        audio = AudioSegment.from_file(out)
        chunks = silence.split_on_silence(audio, min_silence_len=300, silence_thresh=-40, keep_silence=100)
        combined = AudioSegment.empty()
        for c in chunks: combined += c
        combined.export(final, format="wav")
    else:
        os.rename(out, final)
    return final

demo = gr.Interface(fn=process, inputs=[gr.Textbox(label="Text"), gr.Audio(label="Ref Voice", type="filepath"), gr.Checkbox(label="Silence Remover", value=True)], outputs=gr.Audio(label="Output"))
demo.launch(share=True)
''')

!python3.10 app_fixed.py

In [ ]:
# @title 📂 Step 3: Push Output to GitHub Folder
TOKEN = "YOUR_GITHUB_TOKEN"
USER = "shriramnag"
REPO = "TurboVoiceCloner"

!git config --global user.email "your-email@example.com"
!git config --global user.name "{USER}"

# GitHub Folder logic
!mkdir -p outputs
!cp final_voice.wav outputs/
!git add .
!git commit -m "New Cloned Voice Added to Outputs Folder"
!git push https://{TOKEN}@github.com/{USER}/{REPO}.git main
print("✅ All files pushed to GitHub successfully!")